# Sparkle V2 — Fase 3.5b: tres brazos de proyección de apariencia

**Protocolo pre-registrado:** `Protocolo_Exp2_Proyeccion_Apariencia.md` (2026-07-29).
**Requiere:** `dataset_fase3_5_raw1280.npz` (de 3.5a) y `dataset_fase3_2_alineado.npz` (de 3.2b).

| Brazo | Proyección 1.280 → d | Etiquetas | Rol |
| :--- | :--- | :--- | :--- |
| **R1** | aleatoria fija, la original de 3.2a | no | reproduce el estado actual |
| **R2** | PCA ajustada **solo** con el train del fold | no | comparador no supervisado |
| **R3** | densa entrenable, end-to-end con la etiqueta | sí | la intervención |

**Diseño estadístico (§4 del protocolo).** No se comparan medias de 5 folds — eso dio t ≈ 1,4
en el experimento 1. Se juntan las predicciones **out-of-fold**: cada clip del pool recibe una
predicción de un modelo que no lo vio, y el bootstrap clusteriza sobre los **82 participantes**
del pool (4,3× los 19 de Test). Comparaciones **pareadas** sobre los mismos participantes.

**El Test NO se abre acá.** Todas las decisiones —brazo y d— se toman sobre OOF.

**Entorno:** Colab. GPU útil pero no imprescindible (R3 son modelos chicos).

In [ ]:
# =====================================================================
# SETUP
# =====================================================================
import sys, os, json, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')
assert os.path.isdir(PROJECT_DIR), f'No existe {PROJECT_DIR}'

CACHE_32   = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
CACHE_1280 = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_5_raw1280.npz'
PROJ_ORIG  = f'{PROJECT_DIR}/checkpoints_fase3_2/proyeccion_aleatoria_original.npz'
OUT_DIR    = f'{PROJECT_DIR}/fase3_5_proyeccion'
os.makedirs(OUT_DIR, exist_ok=True)

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

# ------- Parámetros CONGELADOS por el pre-registro -------
SEED       = 42
UMBRAL     = 0.5              # regla 2: no se toca
N_FOLDS    = 5
N_BOOT      = 5000
DS         = [16, 32, 64, 128]   # barrido de cuello de botella, SOLO en OOF
SEEDS_R3   = [0, 1, 2, 3, 4]     # regla 5: R3 no es determinístico
MARGEN_MIN = 0.02             # +2,0 pts de BalAcc: umbral de relevancia práctica (§5)
N_FRAMES, N_EMB_FRAMES = 60, 16
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']
np.random.seed(SEED); tf.random.set_seed(SEED)
print(f'Brazos R1/R2/R3 · d ∈ {DS} · {N_FOLDS} folds · OOF + bootstrap por participante')

## 1. Datos — geometría de la caché de 3.2b, apariencia cruda de 3.5a

In [ ]:
for p in (CACHE_32, CACHE_1280):
    assert os.path.exists(p), f'Falta {p}'
d32, d80 = np.load(CACHE_32, allow_pickle=True), np.load(CACHE_1280, allow_pickle=True)
FEATS22 = [str(f) for f in d32['feature_names']]
DIM_RAW = int(d80['dim_raw'])

def ids_of(a): return np.array([str(c) for c in a])
# La alineación se hereda de 3.5a §7; se verifica igual (barato y evita un desastre silencioso)
for k in ('tr', 'va', 'te'):
    assert (ids_of(d32[f'id_{k}']) == ids_of(d80[f'id_{k}'])).all(), \
        f'Los ClipIDs de {k} no coinciden entre las dos cachés'
print('✅ Alineación de ClipIDs verificada entre la caché de 3.2b y la de 3.5a')

# Pool Train+Validation. Test se carga pero NO se usa en este notebook.
Xg_pool = np.concatenate([d32['Xg_tr'], d32['Xg_va']])
Xr_pool = np.concatenate([d80['Xr_tr'], d80['Xr_va']])
y3_pool = np.concatenate([d32['y3_tr'], d32['y3_va']])
id_pool = np.concatenate([ids_of(d32['id_tr']), ids_of(d32['id_va'])])
y_pool  = (y3_pool > 0).astype(np.int64)
part_pool = np.array([c[:6] for c in id_pool])
N_PART = len(set(part_pool))
print(f'Pool: {len(y_pool)} clips · {N_PART} participantes · apariencia {Xr_pool.shape} '
      f'({Xr_pool.dtype})')
print(f'Trivial del pool: {max((y_pool==0).mean(), (y_pool==1).mean()):.4f}')

W_rand, b_rand = (lambda z: (z['W'], z['b']))(np.load(PROJ_ORIG)) if os.path.exists(PROJ_ORIG) \
                 else (None, None)
assert W_rand is not None, (f'Falta {PROJ_ORIG}. Lo genera 3.5a §3 y es imprescindible: sin la '
                            'proyección aleatoria ORIGINAL, R1 no reproduce el estado actual.')
print(f'Proyección aleatoria original cargada: {W_rand.shape}')

## 2. Preprocesamiento — idéntico a 3.3, ajustado por fold

In [ ]:
def interpolate_sequence(seq):
    df = pd.DataFrame(seq).interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean(X, medians=None, thresh=0.30):
    keep, out = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > thresh:
            continue
        s = interpolate_sequence(X[i])
        if np.isnan(s).any() and medians is not None:
            idx = np.where(np.isnan(s)); s[idx] = np.take(medians, idx[1])
        out.append(s); keep.append(i)
    return np.stack(out), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X):
    d_ = np.zeros_like(X[:, :, delta_idx]); d_[:, 1:, :] = np.diff(X[:, :, delta_idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

def resumir(X):
    X = np.nan_to_num(X)
    return np.hstack([X.mean(axis=1), X.std(axis=1)])

def metricas(y, p, umbral=UMBRAL):
    pred = (p > umbral).astype(int)
    triv = max((y == 0).mean(), (y == 1).mean())
    return {'n': int(len(y)), 'trivial': round(float(triv), 4),
            'accuracy': round(float(accuracy_score(y, pred)), 4),
            'balanced_acc': round(float(balanced_accuracy_score(y, pred)), 4),
            'f1_macro': round(float(f1_score(y, pred, average='macro')), 4),
            'brecha': round(float(accuracy_score(y, pred) - triv), 4),
            'auc': round(float(roc_auc_score(y, p)), 4)}

## 3. Los tres brazos

R1 y R2 comparten cabeza (`LogisticRegression`). R3 entrena la proyección junto con una cabeza
lineal, y además se corre **R3-lineal**: se congela la proyección aprendida y se ajusta la
cabeza con `LogisticRegression`, para tener una comparación con cabeza idéntica a R1/R2
(§3 del protocolo — el único desvío inevitable de "protocolo igualado").

In [ ]:
def cabeza_logreg(R_tr, y_tr, R_ev):
    sc = StandardScaler().fit(R_tr)
    lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
    lr.fit(sc.transform(R_tr), y_tr)
    return lr.predict_proba(sc.transform(R_ev))[:, 1]

def brazo_R1(Er_tr, Er_ev, Rg_tr, Rg_ev, y_tr, d):
    '''Proyección aleatoria ORIGINAL, congelada. d se ignora salvo para recortar columnas:
    con d < 128 se toman las primeras d, que siguen siendo direcciones aleatorias.'''
    P = lambda E: (E @ W_rand[:, :d] + b_rand[:d])
    return cabeza_logreg(np.hstack([Rg_tr, resumir(P(Er_tr))]), y_tr,
                         np.hstack([Rg_ev, resumir(P(Er_ev))]))

def brazo_R2(Er_tr, Er_ev, Rg_tr, Rg_ev, y_tr, d):
    '''PCA ajustada SOLO con los frames del train del fold. No usa etiquetas.'''
    pca = PCA(n_components=d, random_state=SEED).fit(
        Er_tr.reshape(-1, DIM_RAW).astype(np.float32))
    P = lambda E: pca.transform(E.reshape(-1, DIM_RAW).astype(np.float32)
                                ).reshape(len(E), N_EMB_FRAMES, d)
    return cabeza_logreg(np.hstack([Rg_tr, resumir(P(Er_tr))]), y_tr,
                         np.hstack([Rg_ev, resumir(P(Er_ev))]))

def build_R3(d, n_geom_res, dropout=0.3, l2=1e-4):
    '''Proyección entrenable por frame -> resumen temporal -> concat con geometría -> lineal.'''
    reg = keras.regularizers.l2(l2)
    inp_e = keras.Input((N_EMB_FRAMES, DIM_RAW), name='emb_raw')
    inp_r = keras.Input((n_geom_res,), name='resumen_geom')
    z = layers.Dropout(dropout, name='emb_drop')(inp_e)
    z = layers.Dense(d, activation=None, kernel_regularizer=reg, name='proyeccion')(z)
    z = layers.Concatenate(name='resumen_emb')(
        [layers.GlobalAveragePooling1D(name='mean_t')(z),
         layers.Lambda(lambda t: tf.math.reduce_std(t, axis=1), name='std_t')(z)])
    h = layers.Concatenate(name='concat')([inp_r, z])
    h = layers.BatchNormalization(name='bn')(h)
    out = layers.Dense(1, activation='sigmoid', kernel_regularizer=reg, name='binary')(h)
    return keras.Model([inp_e, inp_r], out, name=f'R3_d{d}')

def brazo_R3(Er_tr, Er_ev, Rg_tr, Rg_ev, y_tr, d, seed, part_tr, devolver_proyeccion=False):
    keras.utils.set_random_seed(seed)
    # 10 % del train del fold como validación interna para el early stopping.
    # Se corta por PARTICIPANTE para no filtrar (regla 7).
    rng = np.random.default_rng(seed)
    ptr = part_tr
    uniq = np.unique(ptr); rng.shuffle(uniq)
    val_p = set(uniq[:max(1, int(0.1 * len(uniq)))])
    m_va = np.array([p in val_p for p in ptr]); m_tr = ~m_va
    assert not (set(ptr[m_tr]) & set(ptr[m_va])), 'LEAKAGE en el split interno de R3'

    gsc = StandardScaler().fit(Rg_tr[m_tr])
    esc_mu = np.nanmean(Er_tr[m_tr].reshape(-1, DIM_RAW).astype(np.float32), axis=0)
    esc_sd = np.nanstd(Er_tr[m_tr].reshape(-1, DIM_RAW).astype(np.float32), axis=0) + 1e-6
    norm_e = lambda E: ((np.nan_to_num(E.astype(np.float32)) - esc_mu) / esc_sd)

    cw = {0: float(len(y_tr) / (2 * max((y_tr == 0).sum(), 1))),
          1: float(len(y_tr) / (2 * max((y_tr == 1).sum(), 1)))}
    m = build_R3(d, Rg_tr.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1))
    m.fit([norm_e(Er_tr[m_tr]), gsc.transform(Rg_tr[m_tr])], y_tr[m_tr],
          validation_data=([norm_e(Er_tr[m_va]), gsc.transform(Rg_tr[m_va])], y_tr[m_va]),
          epochs=80, batch_size=128, class_weight=cw, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                                   restore_best_weights=True),
                     keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                       patience=5, min_lr=1e-5)])
    p_ev = m.predict([norm_e(Er_ev), gsc.transform(Rg_ev)], batch_size=256, verbose=0).ravel()
    if not devolver_proyeccion:
        return p_ev
    Wp, bp = [np.asarray(w) for w in m.get_layer('proyeccion').get_weights()]
    P = lambda E: (norm_e(E) @ Wp + bp)
    p_lin = cabeza_logreg(np.hstack([Rg_tr, resumir(P(Er_tr))]), y_tr,
                          np.hstack([Rg_ev, resumir(P(Er_ev))]))
    return p_ev, p_lin

## 4. Bucle OOF — cada clip predicho por un modelo que no lo vio

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(sgkf.split(Xg_pool, y_pool, groups=part_pool))

oof = {}          # (brazo, d) -> array de probabilidades, NaN donde el clip fue descartado
mask_ok = np.zeros(len(y_pool), dtype=bool)
def slot(clave):
    if clave not in oof:
        oof[clave] = np.full(len(y_pool), np.nan)
    return oof[clave]

t0 = time.time()
for fold, (tr, va) in enumerate(folds):
    assert not (set(part_pool[tr]) & set(part_pool[va])), f'LEAKAGE fold {fold}'   # regla 7

    g_med = np.nanmedian(Xg_pool[tr].reshape(-1, 22), axis=0)
    Xg_a, kga = clean(Xg_pool[tr], g_med); Xg_b, kgb = clean(Xg_pool[va], g_med)
    Er_a, kea = clean(Xr_pool[tr].astype(np.float32), None, thresh=0.5)
    Er_b, keb = clean(Xr_pool[va].astype(np.float32), None, thresh=0.5)
    ka = np.intersect1d(kga, kea); kb = np.intersect1d(kgb, keb)
    ig = {k: i for i, k in enumerate(kga)}; ie = {k: i for i, k in enumerate(kea)}
    jg = {k: i for i, k in enumerate(kgb)}; je = {k: i for i, k in enumerate(keb)}

    gsc = StandardScaler().fit(add_deltas(Xg_a).reshape(-1, 32))
    sg = lambda X: gsc.transform(X.reshape(-1, 32)).reshape(len(X), N_FRAMES, 32)
    Rg_tr = resumir(sg(add_deltas(Xg_a))[[ig[k] for k in ka]])
    Rg_ev = resumir(sg(add_deltas(Xg_b))[[jg[k] for k in kb]])
    Er_tr = Er_a[[ie[k] for k in ka]]; Er_ev = Er_b[[je[k] for k in kb]]
    y_tr = y_pool[tr][ka]; idx_ev = va[kb]
    mask_ok[idx_ev] = True
    part_fold_tr = part_pool[tr][ka]          # split interno de R3, por participante

    for d in DS:
        slot(('R1', d))[idx_ev] = brazo_R1(Er_tr, Er_ev, Rg_tr, Rg_ev, y_tr, d)
        slot(('R2', d))[idx_ev] = brazo_R2(Er_tr, Er_ev, Rg_tr, Rg_ev, y_tr, d)
        ps, pl = [], []
        for s in SEEDS_R3:
            a, b = brazo_R3(Er_tr, Er_ev, Rg_tr, Rg_ev, y_tr, d, s,
                            part_tr=part_fold_tr, devolver_proyeccion=True)
            ps.append(a); pl.append(b)
            keras.backend.clear_session()      # evita que se acumulen 20 grafos por fold
        slot(('R3', d))[idx_ev] = np.mean(ps, axis=0)          # promedio sobre semillas
        slot(('R3lin', d))[idx_ev] = np.mean(pl, axis=0)
        for s, a in enumerate(ps):
            slot((f'R3_s{s}', d))[idx_ev] = a                  # por semilla, para el σ
    print(f'fold {fold} listo ({len(idx_ev)} clips evaluados) — {(time.time()-t0)/60:.1f} min')

# Assert de cobertura OOF: cada clip que sobrevivió a la limpieza tiene exactamente una predicción
for clave, v in oof.items():
    assert np.isnan(v[mask_ok]).sum() == 0, f'{clave}: hay clips sin predicción OOF'
    assert np.isnan(v[~mask_ok]).all(), f'{clave}: hay predicciones para clips descartados'
print(f'\n✅ OOF completo: {mask_ok.sum()} de {len(y_pool)} clips '
      f'({len(set(part_pool[mask_ok]))} participantes)')

y_oof, part_oof = y_pool[mask_ok], part_pool[mask_ok]
np.savez(f'{OUT_DIR}/oof_predicciones.npz',
         y=y_oof, part=part_oof, ids=id_pool[mask_ok],
         **{f'{b}_d{d}': v[mask_ok] for (b, d), v in oof.items()})

## 5. Resultados OOF y barrido de d

In [ ]:
filas = []
for (b, d), v in sorted(oof.items()):
    if b.startswith('R3_s'):
        continue
    filas.append({'brazo': b, 'd': d, **metricas(y_oof, v[mask_ok])})
df = pd.DataFrame(filas).sort_values(['brazo', 'd'])
print(df.to_string(index=False))
df.to_csv(f'{OUT_DIR}/oof_resultados.csv', index=False)

print('\n--- Balanced accuracy por brazo y d (métrica primaria) ---')
print(df.pivot(index='d', columns='brazo', values='balanced_acc').to_string())

# σ entre semillas de R3 (regla 5)
for d in DS:
    ss = [metricas(y_oof, oof[(f'R3_s{s}', d)][mask_ok])['balanced_acc'] for s in SEEDS_R3]
    print(f'R3 d={d:3d}: BalAcc por semilla {np.mean(ss):.4f} ± {np.std(ss):.4f}')

## 6. Comparación pareada con bootstrap clusterizado por participante

Remuestreo de los 82 participantes con reemplazo; en cada remuestra se computan **los dos
brazos sobre los mismos participantes** y se toma la diferencia. Regla 6: si el IC cruza cero,
la diferencia no existe.

In [ ]:
def _balacc(y, ok):
    '''Balanced accuracy vectorizada = media de los recalls por clase.
    Equivalente a sklearn pero ~50× más rápida, que importa con 5.000 remuestras × 16
    comparaciones. La equivalencia se verifica en el assert de abajo.'''
    n0 = (y == 0).sum(); n1 = (y == 1).sum()
    if n0 == 0 or n1 == 0:
        return np.nan
    return 0.5 * (ok[y == 0].sum() / n0 + ok[y == 1].sum() / n1)

_pred_ref = (oof[('R1', 128)][mask_ok] > UMBRAL).astype(int)
assert abs(_balacc(y_oof, _pred_ref == y_oof) -
           balanced_accuracy_score(y_oof, _pred_ref)) < 1e-12, '_balacc no coincide con sklearn'

def boot_pareado(pa, pb, y=y_oof, part=part_oof, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    parts = np.unique(part)
    idx = {p: np.where(part == p)[0] for p in parts}
    oka = (pa > UMBRAL).astype(int) == y
    okb = (pb > UMBRAL).astype(int) == y
    difs = np.empty(n_boot)
    for i in range(n_boot):
        sel = rng.choice(parts, size=len(parts), replace=True)
        j = np.concatenate([idx[p] for p in sel])
        yj = y[j]
        difs[i] = _balacc(yj, okb[j]) - _balacc(yj, oka[j])
    difs = difs[~np.isnan(difs)]
    lo, hi = np.percentile(difs, [2.5, 97.5])
    return {'delta_balacc': round(float(np.mean(difs)), 4),
            'IC95': [round(float(lo), 4), round(float(hi), 4)],
            'cruza_cero': bool(lo <= 0 <= hi)}

D_REF = 128     # R1 con d=128 es el estado actual exacto
ref = oof[('R1', D_REF)][mask_ok]
comps = []
for b in ['R2', 'R3', 'R3lin']:
    for d in DS:
        r = boot_pareado(ref, oof[(b, d)][mask_ok])
        comps.append({'comparacion': f'{b} d={d}  vs  R1 d=128', **r})
# R3 vs R2 al mismo d: ¿aporta la supervisión sobre la PCA?
for d in DS:
    r = boot_pareado(oof[('R2', d)][mask_ok], oof[('R3', d)][mask_ok])
    comps.append({'comparacion': f'R3 d={d}  vs  R2 d={d}', **r})

df_c = pd.DataFrame(comps)
print(df_c.to_string(index=False))
df_c.to_csv(f'{OUT_DIR}/oof_comparaciones.csv', index=False)

## 7. Veredicto automático contra el criterio pre-registrado (§5 del protocolo)

In [ ]:
r1 = metricas(y_oof, ref)
print(f'R1 (estado actual) — BalAcc OOF: {r1["balanced_acc"]:.4f} | acc {r1["accuracy"]:.4f} '
      f'| trivial {r1["trivial"]:.4f} | AUC {r1["auc"]:.4f}')
print('Referencia de CV del experimento 1 (§9.3 del registro): BalAcc 0,5918 ± 0,0340.')
if abs(r1['balanced_acc'] - 0.5918) > 0.05:
    print('⚠️  R1 se aparta >5 pts de la referencia. Revisar antes de interpretar los otros '
          'brazos: el pipeline puede no ser el mismo.')

gana = df_c[(~df_c.cruza_cero) & (df_c.delta_balacc >= MARGEN_MIN) &
            (df_c.comparacion.str.contains('vs  R1'))]
print('\n================ VEREDICTO ================')
if len(gana) == 0:
    print('❌ NINGÚN brazo supera a R1 con IC que no cruce cero y margen ≥ +2,0 pts.')
    print('   Conclusión pre-registrada: la rama de apariencia NO aporta con este backbone.')
    print('   Siguiente candidato: 2B (fine-tuning del backbone), NO más variantes de proyección.')
    print('   Documentar el fracaso con el mismo detalle que un éxito (regla 10).')
else:
    print('✅ Brazos que cumplen el criterio:')
    print(gana.to_string(index=False))
    mejor = gana.sort_values('delta_balacc', ascending=False).iloc[0]
    print(f'\nMejor: {mejor.comparacion} → {mejor.delta_balacc:+.4f} BalAcc, IC {mejor.IC95}')
    r3_vs_r2 = df_c[df_c.comparacion.str.contains('vs  R2')]
    if (r3_vs_r2.cruza_cero).all():
        print('\n📌 R3 no supera a R2 en ningún d: el hallazgo es que **la proyección aleatoria '
              'era el defecto**, no que haga falta supervisión. Es la conclusión más fuerte y '
              'la más fácil de defender — reportarla así.')
    print('\n⚠️  d quedó SELECCIONADO sobre OOF: su número de OOF es optimistamente sesgado. '
          'El número no sesgado es el del cierre sobre Test, con d ya congelado.')

## 8. Qué hacer con esto

1. **Registrar en `Fase3_Registro_de_Resultados.md`** la tabla de §5, las comparaciones de §6 y
   el veredicto — sea positivo o negativo (regla 10).
2. **Congelar brazo y d** antes de cualquier evaluación de Test, y anotarlos en una enmienda
   fechada del protocolo.
3. **El cierre sobre Test es una pasada única** (§6 del protocolo), con IC bootstrap
   clusterizado por los **19 participantes** de Test. Nunca más un IC i.i.d. sobre 1.638 clips
   (ver §9.6 del registro).
4. Si el veredicto fue negativo: pasar a 2B (fine-tuning del backbone con augmentation y parada
   sobre `val_loss`), medido contra esta misma referencia R1.
5. Beneficio colateral disponible (§8 del protocolo): con `oof_predicciones.npz` se puede
   reestimar el efecto de la agregación temporal del experimento 1 sobre 82 participantes en
   lugar de 5 folds, sin tocar Test.